In [ ]:
import pandas as pd
import numpy as np
import os
import sys

from model_metrics import summarize_model_performance
from model_tuner import loadObjects

sys.path.append("../")

from core.functions import (
    normalize_actor,
    build_actor_interaction_graph,
    add_pairwise_embedding_features,
)

In [ ]:
model_linear_reg = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/952695909840606430/debe55ab62b54c85b860db9506b1eb2b/artifacts/lr_log_fatalities/model.pkl"
)
model_lasso_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/952695909840606430/b8c16d4d3bdd4c5d8c733c99ba95d9b0/artifacts/lasso_log_fatalities/model.pkl"
)
model_xgb_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/952695909840606430/93a3c1984d264e1ead9897ce6fcc678f/artifacts/xgb_log_fatalities/model.pkl"
)
model_cat_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/952695909840606430/1f9f65deb92d41c7971b2f4c439bf776/artifacts/cat_log_fatalities/model.pkl"
)

In [ ]:
X_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_test.parquet"
)

In [ ]:
y_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/y_test_log_fatalities.parquet"
)

In [ ]:
pre = model_linear_reg.estimator.named_steps[
    "preprocess_column_transformer_Preprocessor"
]

feature_names = pre.get_feature_names_out()

In [ ]:
model_linear_reg.estimator.named_steps

In [ ]:
regression_metrics = summarize_model_performance(
    model=[
        model_linear_reg,
        model_lasso_rfe,
        model_xgb_rfe,
        model_cat_rfe,
    ],
    model_title=[
        "Linear Regression",
        "Lasso RFE",
        "XGBoost RFE",
        "CatBoost RFE",
    ],
    X=X_test,
    y=y_test,
    model_type="regression",
    include_adjusted_r2=True,
    return_df=True,
    decimal_places=2,
    overall_only=False,
)

In [ ]:
regression_metrics = regression_metrics.drop(
    columns=["Metric", "Variable", "Coefficient"]
)

In [ ]:
regression_metrics

In [ ]:
len(model_linear_reg.get_feature_names())

In [ ]:
len(model_xgb_rfe.get_feature_names())

In [ ]:
coef_df = pd.DataFrame(
    {
        "feature": model_linear_reg.get_feature_names(),
        "coef": model_linear_reg.estimator.named_steps["lr"].coef_,
    }
).sort_values("coef", key=abs, ascending=False)

coef_df.head(20)

In [ ]:
model_xgb_rfe.estimator

In [ ]:
df = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/raw/acled_ukraine_data_2026_01_02.parquet"
)

In [ ]:
X_test = X_test.assign(
    **pd.get_dummies(X_test[["sub_event_type", "source_scale"]], dtype=int)
)

In [ ]:
[col for col in X_test.columns if "Regional" in col]

In [ ]:
[col for col in X_test.columns if "source" in col]

In [ ]:
[col for col in model_xgb_rfe.get_feature_names() if "source" in col]

In [ ]:
import os

os.getcwd()

In [ ]:
os.path.exists("../data")

In [ ]:
label_map = (None,)
apply_label_map = (True,)


label_replacements = {
    # Sub Event Types
    "Shelling/artillery/missile attack": "Shelling / Missile",
    "Peaceful protest": "Peaceful protest",
    "Mob violence": "Mob violence",
    "Government regains territory": "Gov regains territory",
    "Change to group/activity": "Group change",
    "Armed clash": "Armed clash",
    "Abduction/forced disappearance": "Forced disappearance",
    # Source Scale
    "Subnational-Regional": "Subnational Regional",
    "Subnational": "Subnational",
    "Other-Subnational": "Other Subnational",
    "Other-National": "Other National",
    "National": "National",
    "Other-International": "Other International",
    "International": "International",
    "New media-National": "New media National",
    "New media-International": "New media International",
}

In [ ]:
from model_metrics import plot_3d_pdp

plot_3d_pdp(
    model=model_xgb_rfe.estimator,
    dataframe=X_test,
    feature_names=["sub_event_type", "source_scale"],
    x_label="",
    y_label="",
    # x_label="sub_event_type",
    # y_label="source_scale",
    z_label="Partial Dependence",
    title="3D Partial Dependence Plot of Event Type vs. Source Scale",
    html_file_path="../data",
    # image_filename="3d_pdp",
    html_file_name="3d_pdp.html",
    image_path_svg="../data",
    plot_type="interactive",
    text_wrap=80,
    zoom_out_factor=1.3,
    grid_resolution=35,
    label_fontsize=8,
    tick_fontsize=6,
    title_x=0.38,
    top_margin=20,
    # bottom_margin=120,
    right_margin=80,
    left_margin=70,
    cbar_x=0.9,
    cbar_thickness=25,
    show_modebar=False,
    enable_zoom=True,
    save_plots="html",
    readable_labels=False,
    label_wrap_width=14,
    label_max_words=3,
    label_rotation=35,
    label_replacements=label_replacements,
    label_replacements_regex=False,
)

In [ ]:
from model_metrics import plot_3d_pdp

plot_3d_pdp(
    model=model_xgb_rfe.estimator,
    dataframe=X_test,
    feature_names=["sub_event_type", "source_scale"],
    x_label="",
    y_label="",
    z_label="Partial Dependence",
    title="3D Partial Dependence Plot of Event Type vs. Source Scale",
    html_file_path="../data",
    # image_filename="3d_pdp",
    html_file_name="3d_pdp.html",
    image_path_svg="../data",
    plot_type="static",
    text_wrap=80,
    zoom_out_factor=1.3,
    grid_resolution=35,
    label_fontsize=8,
    tick_fontsize=6,
    title_x=0.38,
    top_margin=20,
    # bottom_margin=120,
    right_margin=80,
    left_margin=70,
    cbar_x=0.9,
    cbar_thickness=25,
    show_modebar=False,
    enable_zoom=True,
    save_plots="html",
    readable_labels=True,
    label_wrap_width=14,
    label_max_words=3,
    label_rotation=90,
    label_replacements=label_replacements,
    label_replacements_regex=False,
)

In [ ]:
from model_metrics import summarize_model_performance

In [ ]:
y_test.rename(columns={"log_fatalities": "actual_log_fatalities"}, inplace=True)

In [ ]:
X_test.head()

In [ ]:
predictions_x_test = pd.Series(
    model_xgb_rfe.predict(X_test), index=X_test.index, name="prediction"
)

In [ ]:
log_preds = predictions_x_test.to_frame(name="log_pred_fatalities")

In [ ]:
log_preds["pred_fatalities"] = round(
    np.expm1(log_preds["log_pred_fatalities"]), 0
).astype(int)

In [ ]:
log_preds_merge = log_preds.join(X_test, how="inner", on="event_id_cnty")

In [ ]:
y_test_actual = pd.DataFrame(
    np.expm1(y_test["actual_log_fatalities"]).round(0).astype(int),
    columns=["actual_log_fatalities"],
).rename(columns={"actual_log_fatalities": "actual_fatalities"})

In [ ]:
y_test_actual

In [ ]:
log_preds_merge = y_test_actual.merge(log_preds_merge, how="inner", on="event_id_cnty")

In [ ]:
log_preds_merge[["actual_fatalities", "pred_fatalities"]]

In [ ]:
from sklearn.metrics import r2_score

r2_score(log_preds_merge["actual_fatalities"], log_preds_merge["pred_fatalities"])

In [ ]:
from eda_toolkit import scatter_fit_plot

scatter_fit_plot(
    df=log_preds_merge,
    x_vars=["actual_fatalities"],
    y_vars=["pred_fatalities"],
    show_legend=True,
    show_plot="subplots",
    subplot_figsize=None,
    label_fontsize=14,
    tick_fontsize=12,
    add_best_fit_line=True,
    scatter_color="#808080",
    show_correlation=True,
)

In [ ]:
log_preds_merge

In [ ]:
log_preds_merge.to_csv(
    os.path.join("..", "data", "processed", "log_preds_merge.csv"), index=False
)

In [ ]:
model_xgb_rfe.get_feature_names()

In [ ]:
dir(model_xgb_rfe)